# License Plate Recognition Evaluation

Notebook ini menyimpan analisis hasil batch inference, tabel akurasi bergaya paper, dan grafik ringkas.


## Accuracy Formula

```text
accuracy rate (%) = right data total / data total * 100%
```


## Generated Charts

![Accuracy by Category](../outputs/evaluation/kaggle_ocr_500/accuracy_by_category.png)

![Validity Status Distribution](../outputs/evaluation/kaggle_ocr_500/validity_status_distribution.png)

![Detection Confidence Histogram](../outputs/evaluation/kaggle_ocr_500/detection_confidence_histogram.png)


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

predictions_path = Path('outputs/evaluation/kaggle_ocr_500/results.csv')
details_path = Path('outputs/evaluation/kaggle_ocr_500/paper_style_details.csv')
summary_path = Path('outputs/evaluation/kaggle_ocr_500/paper_style_summary.csv')

predictions = pd.read_csv(predictions_path)
details = pd.read_csv(details_path)
summary = pd.read_csv(summary_path)
summary


In [ ]:
total = len(details)
detected = int(details['detected'].sum())
plate_eval = int(details['has_plate_ground_truth'].sum())
plate_true = int(details['plate_exact'].sum())
validity_eval = int(details['has_validity_ground_truth'].sum())
validity_true = int(details['validity_exact'].sum())

pd.DataFrame([
    {'Metric': 'Detection', 'True': detected, 'Total': total, 'Accuracy Rate': detected / total * 100 if total else 0},
    {'Metric': 'Plate exact', 'True': plate_true, 'Total': plate_eval, 'Accuracy Rate': plate_true / plate_eval * 100 if plate_eval else 0},
    {'Metric': 'Validity exact', 'True': validity_true, 'Total': validity_eval, 'Accuracy Rate': validity_true / validity_eval * 100 if validity_eval else 0},
])


In [ ]:
ax = summary.plot.bar(x='Characters', y='Accuracy Rate', legend=False, color=['#2f6f8f', '#5f8f3f', '#a45b3f'])
ax.set_ylim(0, 100)
ax.set_ylabel('Accuracy Rate (%)')
ax.set_title('Accuracy by Category')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%')
plt.tight_layout()


In [ ]:
status_counts = predictions['validity_status'].fillna('unknown').value_counts()
ax = status_counts.plot.pie(autopct='%1.1f%%', ylabel='', title='Validity Status Distribution')
plt.tight_layout()


In [ ]:
conf = pd.to_numeric(predictions['detection_confidence'], errors='coerce').dropna()
ax = conf.plot.hist(bins=20, color='#4f6f9f')
ax.set_title('YOLO Detection Confidence Distribution')
ax.set_xlabel('Confidence')
plt.tight_layout()


In [ ]:
mismatches = details[details['has_plate_ground_truth'] & ~details['plate_exact']]
cols = ['image', 'plate_expected_norm', 'plate_predicted_norm', 'validity_expected_norm', 'validity_predicted_norm', 'detection_confidence', 'error']
mismatches[cols].head(30)


## Discussion

Kesalahan OCR paling sering muncul pada plate miring, glare, baut/protector yang menutup karakter, font non-standar, serta hasil crop yang masih menyertakan border atau area luar plate. KNN OCR dipertahankan sebagai baseline sesuai artikel, tetapi batch besar ini menunjukkan bahwa tahap OCR perlu ditingkatkan untuk penggunaan real-world.
